# T0: teacher bake-off, SAM models of different sizes (zero-shot)
Which SAM is the best teacher before any fine-tuning? Every model gets the same box prompts (one box per GT component)
on the 348 Crack500 val crops, and its masks are compared with the GT.

* **SAM 2.1:** tiny, small, base_plus, large. **SAM 1:** ViT-B, ViT-L, ViT-H. (SAM 3 is in T2; it needs a separate install.)
* **Metrics:** pixel precision / recall / F1, mean IoU, thin-crack recall, parameters, ms per image.
* **Next:** fine-tune the best one (T1) and compare it with its zero-shot version.

**Attach:** `distill-datasetforme`. **GPU:** T4. **Time:** ~20 min. **Download:** `runs/teacher_bakeoff/bakeoff_valcrop.json`.


In [1]:
!mkdir -p scripts checkpoints runs
!pip install -q SAM-2 || pip install -q git+https://github.com/facebookresearch/sam2.git
!pip install -q segment-anything || pip install -q git+https://github.com/facebookresearch/segment-anything.git


ERROR: Could not find a version that satisfies the requirement SAM-2 (from versions: none)
ERROR: No matching distribution found for SAM-2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
%%writefile scripts/sam2_teacher_finetune.py
#!/usr/bin/env python3
"""
Check and fine-tune the SAM 2.1 teacher on cracks, with the same box prompts used to cache the KD logits.

  eval      box-prompted teacher vs GT on a Crack500 crop split: pixel P/R/F1, mean IoU, thin-crack recall
  finetune  train the mask decoder + prompt encoder on traincrop (image encoder frozen), keep the best val F1
  logits    re-cache teacher logits: crack500_<stem>_logits.npy, M x 256 x 256, same instances and order as
            scripts/generate_teacher_logits.py (connected components >= 50 px, one box each)

Instances are the GT connected components (>= 50 px); the prediction for an image is the union of its instance masks.

Usage:
  python scripts/sam2_teacher_finetune.py eval --crack500 data/datasets/crack500 --split valcrop
  python scripts/sam2_teacher_finetune.py finetune --crack500 data/datasets/crack500 --out runs/teacher_ft
  python scripts/sam2_teacher_finetune.py finetune --train-list data/datasets/crack500_yolo/train_photos25.txt
  python scripts/sam2_teacher_finetune.py eval --split valcrop --weights runs/teacher_ft/sam2_crack_decoder.pt
  python scripts/sam2_teacher_finetune.py logits --weights runs/teacher_ft/sam2_crack_decoder.pt --out data/teacher_logits_ft
"""

import argparse
import json
import math
import random
import time
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn.functional as F

SAM2_CFG = "configs/sam2.1/sam2.1_hiera_l.yaml"
# SAM 2.1 sizes: name -> (config, checkpoint file name on dl.fbaipublicfiles.com/segment_anything_2/092824/)
SAM2_MODELS = {
    "sam2.1_tiny": ("configs/sam2.1/sam2.1_hiera_t.yaml", "sam2.1_hiera_tiny.pt"),
    "sam2.1_small": ("configs/sam2.1/sam2.1_hiera_s.yaml", "sam2.1_hiera_small.pt"),
    "sam2.1_base_plus": ("configs/sam2.1/sam2.1_hiera_b+.yaml", "sam2.1_hiera_base_plus.pt"),
    "sam2.1_large": (SAM2_CFG, "sam2.1_hiera_large.pt"),
}
MIN_AREA = 50
THIN_PX = 4   # a GT pixel is "thin" if the local crack width around it is <= 4 px


def load_predictor(ckpt, weights=None, device=None, cfg=SAM2_CFG):
    from sam2.build_sam import build_sam2
    from sam2.sam2_image_predictor import SAM2ImagePredictor
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    predictor = SAM2ImagePredictor(build_sam2(cfg, str(ckpt), device=device))
    if weights:
        state = torch.load(weights, map_location=device)
        predictor.model.sam_mask_decoder.load_state_dict(state["sam_mask_decoder"])
        predictor.model.sam_prompt_encoder.load_state_dict(state["sam_prompt_encoder"])
    predictor.model.eval()
    return predictor


def read_stems(list_file):
    """Stems from a file of image paths or stems, one per line (e.g. train_photos25.txt from make_train_subset.py)."""
    return {Path(line.strip()).stem for line in Path(list_file).read_text().splitlines() if line.strip()}


def pairs(split_dir, max_images=None, stems=None):
    out = [(p, p.with_suffix(".png")) for p in sorted(Path(split_dir).glob("*.jpg")) if p.with_suffix(".png").exists()]
    if stems is not None:
        out = [x for x in out if x[0].stem in stems]
    return out[:max_images] if max_images else out


def read_pair(img_path, mask_path):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    gt = (cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE) > 127).astype(np.uint8)
    return img, gt


def instances(gt):
    """GT connected components >= MIN_AREA px and their xyxy boxes, in label order (as generate_teacher_logits.py)."""
    n, lab = cv2.connectedComponents(gt)
    masks, boxes = [], []
    for i in range(1, n):
        m = (lab == i).astype(np.uint8)
        if m.sum() < MIN_AREA:
            continue
        ys, xs = np.nonzero(m)
        masks.append(m)
        boxes.append([xs.min(), ys.min(), xs.max(), ys.max()])
    return masks, np.array(boxes, dtype=np.float32).reshape(-1, 4)


def set_image(predictor, img):
    with torch.autocast("cuda", dtype=torch.float16, enabled=predictor.device.type == "cuda"):
        predictor.set_image(img)
    f = predictor._features
    predictor._features = {"image_embed": f["image_embed"].float(), "high_res_feats": [x.float() for x in f["high_res_feats"]]}


def decode(predictor, boxes):
    """Box prompts -> (low-res 256x256 logits, full-res logits), both N x 1 x H x W. Same path as
    SAM2ImagePredictor._predict, but differentiable."""
    _, _, _, box = predictor._prep_prompts(None, None, boxes, None, normalize_coords=True)
    labels = torch.tensor([[2, 3]], dtype=torch.int, device=box.device).repeat(box.shape[0], 1)
    model = predictor.model
    sparse, dense = model.sam_prompt_encoder(points=(box.reshape(-1, 2, 2), labels), boxes=None, masks=None)
    low, _, _, _ = model.sam_mask_decoder(
        image_embeddings=predictor._features["image_embed"],
        image_pe=model.sam_prompt_encoder.get_dense_pe(),
        sparse_prompt_embeddings=sparse, dense_prompt_embeddings=dense,
        multimask_output=False, repeat_image=box.shape[0] > 1,
        high_res_features=predictor._features["high_res_feats"])
    return low, predictor._transforms.postprocess_masks(low, predictor._orig_hw[-1])


def thin_mask(gt):
    dist = cv2.distanceTransform(gt, cv2.DIST_L2, 3)
    local_width = 2 * cv2.dilate(dist, np.ones((5, 5), np.uint8))
    return gt.astype(bool) & (local_width <= THIN_PX)


def accumulate(acc, pred, gt):
    """Add one image (pred: bool HxW, gt: uint8 HxW) to the running pixel counts."""
    g, thin = gt.astype(bool), thin_mask(gt)
    acc["tp"] += int((pred & g).sum()); acc["fp"] += int((pred & ~g).sum()); acc["fn"] += int((~pred & g).sum())
    acc["thin_tp"] += int((pred & thin).sum()); acc["thin_n"] += int(thin.sum())
    union = (pred | g).sum()
    acc["ious"].append(float((pred & g).sum() / union) if union else 1.0)
    return acc


def new_acc():
    return {"tp": 0, "fp": 0, "fn": 0, "thin_tp": 0, "thin_n": 0, "ious": []}


def summarize(acc):
    tp, fp, fn = acc["tp"], acc["fp"], acc["fn"]
    p, r = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    return {"images": len(acc["ious"]), "precision": p, "recall": r, "f1": 2 * p * r / max(p + r, 1e-9),
            "mean_iou": float(np.mean(acc["ious"])), "thin_recall": acc["thin_tp"] / max(acc["thin_n"], 1)}


@torch.no_grad()
def evaluate(predictor, split_dir, max_images=None):
    acc = new_acc()
    for img_path, mask_path in pairs(split_dir, max_images):
        img, gt = read_pair(img_path, mask_path)
        _, boxes = instances(gt)
        pred = np.zeros(gt.shape, bool)
        if len(boxes):
            set_image(predictor, img)
            pred = (decode(predictor, boxes)[1][:, 0] > 0).any(0).cpu().numpy()
        accumulate(acc, pred, gt)
    return summarize(acc)


def jitter(boxes, h, w, frac=0.05):
    bw, bh = boxes[:, 2] - boxes[:, 0], boxes[:, 3] - boxes[:, 1]
    noise = np.random.uniform(-frac, frac, boxes.shape) * np.stack([bw, bh, bw, bh], 1)
    return np.clip(boxes + noise, 0, [w - 1, h - 1, w - 1, h - 1]).astype(np.float32)


def finetune(predictor, train_dir, val_dir, out, epochs=6, lr=1e-4, accum=4, max_train=None, max_val=None, seed=0,
             stems=None):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    out = Path(out); out.mkdir(parents=True, exist_ok=True)
    model = predictor.model
    params = [*model.sam_mask_decoder.parameters(), *model.sam_prompt_encoder.parameters()]
    for p in model.parameters():
        p.requires_grad_(False)
    for p in params:
        p.requires_grad_(True)
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    train = pairs(train_dir, max_train, stems)
    print(f"[finetune] {len(train)} train crops", flush=True)
    steps = epochs * math.ceil(len(train) / accum)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: 0.5 * (1 + math.cos(math.pi * min(s / steps, 1))))

    history = [{"epoch": 0, **evaluate(predictor, val_dir, max_val)}]
    best = history[0]["f1"]
    print(f"[epoch 0] zero-shot val {history[0]}", flush=True)
    for epoch in range(1, epochs + 1):
        model.sam_mask_decoder.train(); model.sam_prompt_encoder.train()
        random.shuffle(train)
        t0, losses = time.time(), []
        opt.zero_grad()
        for i, (img_path, mask_path) in enumerate(train, 1):
            img, gt = read_pair(img_path, mask_path)
            if random.random() < 0.5:
                img, gt = img[:, ::-1], gt[:, ::-1]
            if random.random() < 0.5:
                img, gt = img[::-1], gt[::-1]
            img, gt = np.ascontiguousarray(img), np.ascontiguousarray(gt)
            masks, boxes = instances(gt)
            if not masks:
                continue
            set_image(predictor, img)
            logits = decode(predictor, jitter(boxes, *gt.shape))[1][:, 0]
            target = torch.as_tensor(np.stack(masks), dtype=torch.float32, device=logits.device)
            prob = logits.sigmoid()
            dice = 1 - (2 * (prob * target).sum((1, 2)) + 1) / (prob.sum((1, 2)) + target.sum((1, 2)) + 1)
            loss = F.binary_cross_entropy_with_logits(logits, target) + dice.mean()
            (loss / accum).backward()
            losses.append(loss.item())
            if i % accum == 0 or i == len(train):
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                opt.step(); sched.step(); opt.zero_grad()
            if i % 200 == 0:
                print(f"  epoch {epoch} {i}/{len(train)} loss {np.mean(losses[-200:]):.4f}", flush=True)
        model.eval()
        row = {"epoch": epoch, "train_loss": float(np.mean(losses)), "minutes": (time.time() - t0) / 60,
               **evaluate(predictor, val_dir, max_val)}
        history.append(row)
        print(f"[epoch {epoch}] {row}", flush=True)
        if row["f1"] > best:
            best = row["f1"]
            torch.save({"sam_mask_decoder": model.sam_mask_decoder.state_dict(),
                        "sam_prompt_encoder": model.sam_prompt_encoder.state_dict(), "epoch": epoch, "val": row},
                       out / "sam2_crack_decoder.pt")
        (out / "history.json").write_text(json.dumps(history, indent=2) + "\n")
    return history


@torch.no_grad()
def cache_logits(predictor, img_dir, out_dir, prefix="crack500_", max_images=None):
    """Same instances, prompts and output format as generate_teacher_logits.py --prompt-type box."""
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    written = 0
    for img_path, mask_path in pairs(img_dir, max_images):
        img, gt = read_pair(img_path, mask_path)
        _, boxes = instances(gt)
        if not len(boxes):
            continue
        predictor.set_image(img)
        logits = [predictor.predict(box=b, multimask_output=False)[2][0].astype(np.float32) for b in boxes]
        np.save(out_dir / f"{prefix}{img_path.stem}_logits.npy", np.stack(logits))
        written += 1
    return written


def main():
    ap = argparse.ArgumentParser(description=__doc__.split("\n\n")[0])
    ap.add_argument("cmd", choices=["eval", "finetune", "logits"])
    ap.add_argument("--crack500", default="data/datasets/crack500")
    ap.add_argument("--ckpt", default="checkpoints/sam2_hiera_large.pt")
    ap.add_argument("--model", default="sam2.1_large", choices=list(SAM2_MODELS), help="SAM 2.1 size (must match --ckpt)")
    ap.add_argument("--weights", default=None, help="fine-tuned decoder (from finetune); omit for zero-shot")
    ap.add_argument("--split", default="valcrop")
    ap.add_argument("--out", default="runs/teacher_ft")
    ap.add_argument("--epochs", type=int, default=6)
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--train-list", default=None, help="finetune only on these crops (file of image paths or stems)")
    ap.add_argument("--max", type=int, default=None, help="limit images (smoke tests)")
    ap.add_argument("--max-val", type=int, default=None)
    a = ap.parse_args()
    root = Path(a.crack500)
    predictor = load_predictor(a.ckpt, a.weights, cfg=SAM2_MODELS[a.model][0])
    if a.cmd == "eval":
        res = {"split": a.split, "weights": a.weights or "zero-shot", **evaluate(predictor, root / a.split, a.max)}
        print(json.dumps(res, indent=2))
        Path(a.out).mkdir(parents=True, exist_ok=True)
        tag = Path(a.weights).stem if a.weights else "zeroshot"
        (Path(a.out) / f"eval_{tag}_{a.split}.json").write_text(json.dumps(res, indent=2) + "\n")
    elif a.cmd == "finetune":
        finetune(predictor, root / "traincrop", root / "valcrop", a.out, a.epochs, a.lr, max_train=a.max, max_val=a.max_val,
                 stems=read_stems(a.train_list) if a.train_list else None)
    else:
        print(f"[logits] wrote {cache_logits(predictor, root / 'traincrop', a.out, max_images=a.max)} files to {a.out}")


if __name__ == "__main__":
    main()


Writing scripts/sam2_teacher_finetune.py


In [3]:
%%writefile scripts/sam_bakeoff.py
#!/usr/bin/env python3
"""
Teacher bake-off: zero-shot SAM models of different sizes on Crack500 val crops, all with the same box prompts
(one box per GT component >= 50 px, as used for the KD logits).

  SAM 2.1: tiny, small, base_plus, large        SAM 1: vit_b, vit_l, vit_h   (needs the segment-anything package)

Per model: pixel precision / recall / F1, mean IoU, thin-crack recall, parameters, ms per image.
Checkpoints are downloaded to checkpoints/ when missing. A model that fails is reported and skipped.

Usage:
  python scripts/sam_bakeoff.py --crack500 data/datasets/crack500 --out runs/teacher_bakeoff
  python scripts/sam_bakeoff.py --models sam2.1_tiny sam2.1_large --max 20
"""

import argparse
import json
import sys
import time
import urllib.request
from pathlib import Path

import numpy as np
import torch

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
from scripts.sam2_teacher_finetune import (SAM2_MODELS, accumulate, decode, instances, load_predictor, new_acc, pairs,
                                           read_pair, set_image, summarize)

SAM2_URL = "https://dl.fbaipublicfiles.com/segment_anything_2/092824/"
SAM1_URL = "https://dl.fbaipublicfiles.com/segment_anything/"
# Hugging Face copies, used when the Meta host fails (its TLS certificate was expired on 2026-11-12)
SAM2_HF = "https://huggingface.co/facebook/{repo}/resolve/main/"
SAM1_HF = "https://huggingface.co/ybelkada/segment-anything/resolve/main/checkpoints/"
SAM1_MODELS = {"sam1_vit_b": ("vit_b", "sam_vit_b_01ec64.pth"), "sam1_vit_l": ("vit_l", "sam_vit_l_0b3195.pth"),
               "sam1_vit_h": ("vit_h", "sam_vit_h_4b8939.pth")}
MODELS = [*SAM2_MODELS, *SAM1_MODELS]


def fetch(urls, dst, min_bytes=3e7):
    """Download the first working url to dst unless a complete file is already there; a failed download must not
    leave an empty file."""
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    for url in urls:
        if dst.exists() and dst.stat().st_size >= min_bytes:
            break
        dst.unlink(missing_ok=True)
        print(f"  downloading {url}", flush=True)
        try:
            urllib.request.urlretrieve(url, dst)
        except Exception as e:
            print(f"  download failed ({type(e).__name__}: {e})", flush=True)
    assert dst.exists() and dst.stat().st_size >= min_bytes, f"could not download {dst.name}"
    return dst


def sam2_mask_fn(name, ckpt_dir, device):
    cfg, fname = SAM2_MODELS[name]
    # the large checkpoint is stored under its historical name by the other scripts
    old = Path(ckpt_dir) / "sam2_hiera_large.pt"
    ckpt = old if name == "sam2.1_large" and old.exists() and old.stat().st_size > 8e8 else fetch(
        [SAM2_URL + fname, SAM2_HF.format(repo=fname[:-3].replace("_", "-")) + fname], Path(ckpt_dir) / fname)
    predictor = load_predictor(ckpt, device=device, cfg=cfg)

    def mask_fn(img, boxes):
        set_image(predictor, img)
        return (decode(predictor, boxes)[1][:, 0] > 0).any(0).cpu().numpy()
    return mask_fn, predictor.model


def sam1_mask_fn(name, ckpt_dir, device):
    from segment_anything import SamPredictor, sam_model_registry
    arch, fname = SAM1_MODELS[name]
    model = sam_model_registry[arch](checkpoint=str(fetch([SAM1_URL + fname, SAM1_HF + fname], Path(ckpt_dir) / fname))).to(device).eval()
    predictor = SamPredictor(model)

    def mask_fn(img, boxes):
        predictor.set_image(img)
        b = predictor.transform.apply_boxes_torch(torch.as_tensor(boxes, device=device), img.shape[:2])
        masks, _, _ = predictor.predict_torch(point_coords=None, point_labels=None, boxes=b, multimask_output=False)
        return masks[:, 0].any(0).cpu().numpy()
    return mask_fn, model


@torch.no_grad()
def evaluate_fn(mask_fn, split_dir, max_images=None):
    acc, seconds = new_acc(), 0.0
    for img_path, mask_path in pairs(split_dir, max_images):
        img, gt = read_pair(img_path, mask_path)
        _, boxes = instances(gt)
        pred = np.zeros(gt.shape, bool)
        if len(boxes):
            t0 = time.time()
            pred = mask_fn(img, boxes)
            seconds += time.time() - t0
        accumulate(acc, pred, gt)
    return {**summarize(acc), "ms_per_image": 1000 * seconds / max(len(acc["ious"]), 1)}


def main():
    ap = argparse.ArgumentParser(description=__doc__.split("\n\n")[0])
    ap.add_argument("--crack500", default="data/datasets/crack500")
    ap.add_argument("--split", default="valcrop")
    ap.add_argument("--models", nargs="+", default=MODELS, choices=MODELS)
    ap.add_argument("--ckpt-dir", default="checkpoints")
    ap.add_argument("--out", default="runs/teacher_bakeoff")
    ap.add_argument("--max", type=int, default=None, help="limit images (smoke tests)")
    a = ap.parse_args()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    out = Path(a.out); out.mkdir(parents=True, exist_ok=True)
    results = {}
    for name in a.models:
        print(f"[{name}]", flush=True)
        try:
            mask_fn, model = (sam2_mask_fn if name in SAM2_MODELS else sam1_mask_fn)(name, a.ckpt_dir, device)
            results[name] = {"params_M": sum(p.numel() for p in model.parameters()) / 1e6,
                             **evaluate_fn(mask_fn, Path(a.crack500) / a.split, a.max)}
            print(f"  {json.dumps(results[name])}", flush=True)
            del mask_fn, model
            torch.cuda.empty_cache()
        except Exception as e:   # one broken download or install must not lose the other models
            results[name] = {"error": f"{type(e).__name__}: {e}"}
            print(f"  FAILED {results[name]['error']}", flush=True)
        (out / f"bakeoff_{a.split}.json").write_text(json.dumps(results, indent=2) + "\n")
    ok = {k: v for k, v in results.items() if "error" not in v}
    print(f"\n{'model':<18}{'params M':>9}{'ms/img':>8}{'prec':>7}{'recall':>7}{'F1':>7}{'IoU':>7}{'thin R':>7}")
    for k, v in sorted(ok.items(), key=lambda kv: -kv[1]["f1"]):
        print(f"{k:<18}{v['params_M']:>9.1f}{v['ms_per_image']:>8.0f}{v['precision']:>7.3f}{v['recall']:>7.3f}"
              f"{v['f1']:>7.3f}{v['mean_iou']:>7.3f}{v['thin_recall']:>7.3f}")
    if ok:
        print(f"\n[Best zero-shot teacher by pixel F1] {max(ok, key=lambda k: ok[k]['f1'])}")


if __name__ == "__main__":
    main()


Writing scripts/sam_bakeoff.py


In [4]:
# ── Link raw Crack500 (the folder holding traincrop/ valcrop/ testcrop/ with .jpg + .png masks) ──
import os, json, sys
from pathlib import Path
sys.path.insert(0, ".")
raw = sorted(Path(r) for r, d, f in os.walk("/kaggle/input", followlinks=True) if "traincrop" in d)
assert raw, "[FATAL] raw Crack500 (a folder containing traincrop/) not found in /kaggle/input: attach distill-datasetforme"
CRACK500 = raw[0]
counts = {s: len(list((CRACK500 / s).glob("*.jpg"))) for s in ['traincrop', 'valcrop', 'testcrop']}
print(f"[Dataset] {CRACK500} {counts}")
assert counts == {'traincrop': 1896, 'valcrop': 348, 'testcrop': 1124}, f"[FATAL] not the canonical Crack500 crops: {counts}"


[Dataset] /kaggle/input/datasets/rauffatali/distill-datasetforme/datasets/crack500 {'traincrop': 1896, 'valcrop': 348, 'testcrop': 1124}


In [5]:
!python scripts/sam_bakeoff.py --crack500 {CRACK500} --split valcrop --out runs/teacher_bakeoff


[sam2.1_tiny]
  downloading https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt
  download failed (URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: certificate has expired (_ssl.c:1032)>)
  downloading https://huggingface.co/facebook/sam2.1-hiera-tiny/resolve/main/sam2.1_hiera_tiny.pt
  {"params_M": 38.962498, "images": 348, "precision": 0.3608048021299506, "recall": 0.8039117081025542, "f1": 0.498070049189894, "mean_iou": 0.4891382670258444, "thin_recall": 0.615567036503769, "ms_per_image": 116.89863328276009}
[sam2.1_small]
  downloading https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt
  download failed (URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: certificate has expired (_ssl.c:1032)>)
  downloading https://huggingface.co/facebook/sam2.1-hiera-small/resolve/main/sam2.1_hiera_small.pt
  {"params_M": 46.060354, "images": 348, "precision": 0.3871879767503